# Extract CLIP ViT-L/14 Video Features

从游戏视频中提取 CLIP ViT-L/14 视觉特征（768-dim），替代原有的 ResNet-50（2048-dim）。

**输出：**
- `features/video_clip/` — 原始 CLIP 特征 (8fps)
- `features/aligned/video_clip/` — 对齐到 5Hz 的 CLIP 特征

**运行要求：** A100 或 H100 GPU

In [ ]:
# Cell 0: GPU 检查
!nvidia-smi

In [ ]:
# Cell 1: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: 克隆项目仓库
from getpass import getpass
import os

token = getpass('Enter your GitHub Personal Access Token: ')
user_repo = 'szaaaaaa/ProjectExperiment'

if os.path.exists('/content/ProjectExperiment'):
    %cd /content/ProjectExperiment
    !git pull
else:
    !git clone https://{token}@github.com/{user_repo}.git /content/ProjectExperiment
    %cd /content/ProjectExperiment

In [ ]:
# Cell 3: 安装依赖
!pip install open_clip_torch -q

In [ ]:
# Cell 4: 定义路径变量
import os

DRIVE = '/content/drive/MyDrive'
AMUCS_RAW = f'{DRIVE}/AmuCS/Affective Multimodal Counter-Strike video game dataset (AMuCS) - Public/researchdata'
AMUCS_EXP = f'{DRIVE}/AmuCS_experiment'

VIDEO_DIR = f'{AMUCS_RAW}/gameplay_videos_nospeech'
VIDEO_CLIP_DIR = f'{AMUCS_EXP}/features/video_clip'
ALIGNED_DIR = f'{AMUCS_EXP}/features/aligned'

# 验证路径
assert os.path.exists(VIDEO_DIR), f'Video dir not found: {VIDEO_DIR}'
assert os.path.exists(ALIGNED_DIR), f'Aligned dir not found: {ALIGNED_DIR}'
print(f'Video dir: {VIDEO_DIR}')
print(f'Output dir: {VIDEO_CLIP_DIR}')
print(f'Aligned dir: {ALIGNED_DIR}')

In [ ]:
# Cell 5: 提取 CLIP ViT-L/14 特征
# A100 约 2-3 小时，H100 约 1.5-2 小时
# 支持断点续传（已存在的 .pt 文件会跳过）
import os
os.chdir('/content/ProjectExperiment')
!python scripts/extract_video_features.py \
  --video_dir "{VIDEO_DIR}" \
  --output_dir "{VIDEO_CLIP_DIR}" \
  --backbone clip_vit_l14 \
  --target_fps 8 \
  --batch_size 64 \
  --device cuda \
  --amp \
  --session_mode subdirs \
  --name_mode amucs

In [ ]:
# Cell 6: 验证提取结果
import glob, torch

pts = sorted(glob.glob(f'{VIDEO_CLIP_DIR}/*.pt'))
print(f'Extracted files: {len(pts)}')

if pts:
    obj = torch.load(pts[0], map_location='cpu', weights_only=False)
    print(f'Sample: {os.path.basename(pts[0])}')
    print(f'  Feature shape: {tuple(obj["features"].shape)}')  # expect [T, 768]
    print(f'  Backbone: {obj["meta"]["backbone"]}')
    print(f'  Feature dim: {obj["meta"]["feature_dim"]}')
    print(f'  Sample FPS: {obj.get("sample_fps", "N/A")}')

In [ ]:
# Cell 7: 对齐 video_clip 到 5Hz
import os
FEAT = f'{AMUCS_EXP}/features'
STAGE = f'{FEAT}/_stage_video_clip'

!mkdir -p "{STAGE}"
!ln -sfn "{FEAT}/video_clip" "{STAGE}/video_clip"

os.chdir('/content/ProjectExperiment')
!python scripts/sync_data.py \
  --input_root "{STAGE}" \
  --output_root "{ALIGNED_DIR}" \
  --modalities video_clip \
  --grid_mode uniform \
  --target_hz 5 \
  --resample nearest \
  --time_origin zero

!rm -rf "{STAGE}"

In [ ]:
# Cell 8: 验证对齐结果
import glob, torch, os

clip_aligned = sorted(glob.glob(f'{ALIGNED_DIR}/video_clip/*.pt'))
video_aligned = sorted(glob.glob(f'{ALIGNED_DIR}/video/*.pt'))
km_aligned = sorted(glob.glob(f'{ALIGNED_DIR}/km/*.pt'))

print(f'Aligned video_clip: {len(clip_aligned)} files')
print(f'Aligned video (original ResNet): {len(video_aligned)} files')
print(f'Aligned km: {len(km_aligned)} files')

# 对比同一个 stem 的长度是否一致
if clip_aligned and video_aligned:
    stem = os.path.basename(clip_aligned[0])
    clip_obj = torch.load(clip_aligned[0], map_location='cpu', weights_only=False)
    # 找到同名的 video 文件
    video_path = f'{ALIGNED_DIR}/video/{stem}'
    if os.path.exists(video_path):
        video_obj = torch.load(video_path, map_location='cpu', weights_only=False)
        print(f'\nSample: {stem}')
        print(f'  CLIP shape:   {tuple(clip_obj["features"].shape)}')   # [T, 768]
        print(f'  ResNet shape: {tuple(video_obj["features"].shape)}')  # [T, 2048]
        print(f'  Length match: {clip_obj["features"].shape[0] == video_obj["features"].shape[0]}')

print('\nDone! CLIP features ready at:', f'{ALIGNED_DIR}/video_clip/')